# Metal Surface Defect Detection

## Aim
Detect scratches and cracks using edge detection, segmentation, thresholding, contours, and morphology.

## Theory
Thresholds segment unusually dark cracks and bright scratches. Morphological closing joins broken responses, opening removes isolated noise, and contours localize candidate defects.

In [ ]:
!pip -q install opencv-python-headless matplotlib numpy pandas

In [ ]:
# Portable setup: local Jupyter, Google Drive, or Google Colab
from pathlib import Path
import os, zipfile

FOLDER_NAME = "04_PRACTICAL_4"
ZIP_NAME = "04_PRACTICAL_4.zip"

def locate_project():
    candidates = [Path.cwd(), Path.cwd()/FOLDER_NAME, Path('/content')/FOLDER_NAME]
    candidates += list(Path('/content').glob('**/' + FOLDER_NAME)) if Path('/content').exists() else []
    for p in candidates:
        if (p/'inputs').exists() and (p/'outputs').exists():
            return p.resolve()
    return None

BASE = locate_project()
if BASE is None:
    try:
        from google.colab import files
        print(f"Upload {ZIP_NAME} when the chooser opens.")
        uploaded = files.upload()
        chosen = next((n for n in uploaded if n.lower().endswith('.zip')), None)
        if chosen:
            with zipfile.ZipFile(chosen) as zf:
                zf.extractall('/content')
            BASE = locate_project()
    except ImportError:
        pass
if BASE is None:
    raise FileNotFoundError(
        f"Could not find {FOLDER_NAME}. Extract {ZIP_NAME}, or upload that ZIP in Colab."
    )
os.chdir(BASE)
INPUTS, OUTPUTS, MODELS = BASE/'inputs', BASE/'outputs', BASE/'models'
OUTPUTS.mkdir(exist_ok=True)
print('Project directory:', BASE)


In [ ]:
import cv2, numpy as np, pandas as pd, matplotlib.pyplot as plt
img=cv2.imread(str(INPUTS/'metal_surface_defects.png'));gt=cv2.imread(str(INPUTS/'ground_truth_mask.png'),0);gray=cv2.cvtColor(img,cv2.COLOR_BGR2GRAY)
edges=cv2.Canny(gray,45,120);dark=cv2.threshold(gray,75,255,cv2.THRESH_BINARY_INV)[1];bright=cv2.threshold(gray,215,255,cv2.THRESH_BINARY)[1]
mask=cv2.bitwise_or(dark,bright);mask=cv2.morphologyEx(mask,cv2.MORPH_CLOSE,cv2.getStructuringElement(cv2.MORPH_ELLIPSE,(7,7)),iterations=2);mask=cv2.morphologyEx(mask,cv2.MORPH_OPEN,np.ones((3,3),np.uint8))
contours,_=cv2.findContours(mask,cv2.RETR_EXTERNAL,cv2.CHAIN_APPROX_SIMPLE);valid=[c for c in contours if cv2.contourArea(c)>40];annotated=img.copy();cv2.drawContours(annotated,valid,-1,(0,0,255),2)
for n,x in [('edges_rerun.png',edges),('mask_rerun.png',mask),('annotated_rerun.png',annotated)]:cv2.imwrite(str(OUTPUTS/n),x)
fig,ax=plt.subplots(1,4,figsize=(16,4));
for a,x,t,c in zip(ax,[img,edges,mask,annotated],['Input','Canny','Morphological mask','Contours'],[None,'gray','gray',None]):a.imshow(cv2.cvtColor(x,cv2.COLOR_BGR2RGB) if x.ndim==3 else x,cmap=c);a.set_title(t);a.axis('off')
plt.show()

### Validation against supplied ground truth

In [ ]:
p=mask>0;t=gt>0;tp=(p&t).sum();fp=(p&~t).sum();fn=(~p&t).sum();precision=tp/max(tp+fp,1);recall=tp/max(tp+fn,1);iou=tp/max(tp+fp+fn,1)
df=pd.DataFrame([[len(valid),precision,recall,iou]],columns=['contours','precision','recall','iou']);display(df);df.to_csv(OUTPUTS/'defect_metrics_rerun.csv',index=False)

## Result and conclusion
Run all cells and compare the generated files in `outputs/` with the supplied reference outputs. The observations and conclusion are also summarized in `OUTPUT_SUMMARY.md`.